In [2]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

In [3]:
import numpy as np
import pandas as pd
import os

Load Data

In [4]:
input_path = "D:/Projects with Git/AI-For-Health-IITGN-SRIP/Dataset"

patient_paths = [f.path for f in os.scandir(input_path) if f.is_file() and f.name.endswith('_dataset.npz')]

datasets = {}

for path in patient_paths:
    patient_id = os.path.basename(path).replace('_dataset.npz', '')

    with np.load(path, allow_pickle=True) as data:
        datasets[patient_id] = {
            "X": data["X"],
            "y_sleep": data["y_sleep"],
            "y_breath": data["y_breath"]
        }

Prepare categorical to numerical mapping

In [5]:
sleep_set = set()
breath_set = set()

for patient in datasets:
    sleep_set.update(np.unique(datasets[patient]["y_sleep"]))
    breath_set.update(np.unique(datasets[patient]["y_breath"]))

sleep_set = sorted(sleep_set)
breath_set = sorted(breath_set)

print("\nUnique Categories")
print(f"Sleep States:{sleep_set}")
print(f"Breathing Irregularities:{breath_set}")

sleep_map = {
    " A": 0,
    " N1": 1,
    " N2": 2,
    " N3": 3,
    " Movement": 4,
    " REM": 5,
    " Wake": 6
}

breath_map = {
    "Body event": 0,
    "Hypopnea": 1,
    "Mixed Apnea": 2,
    "Normal": 3,
    "Obstructive Apnea": 4
}

print("\nError/Miss Check")
print(f"Sleep States: {set(sleep_set) - set(sleep_map.keys())}")
print(f"Breathing Irregularities: {set(breath_set) - set(breath_map.keys())}")


Unique Categories
Sleep States:[' A', ' Movement', ' N1', ' N2', ' N3', ' REM', ' Wake']
Breathing Irregularities:['Body event', 'Hypopnea', 'Mixed Apnea', 'Normal', 'Obstructive Apnea']

Error/Miss Check
Sleep States: set()
Breathing Irregularities: set()


Normalize the columns in X, apply mapping to y_breath and y_sleep.

# Not used
# Normalization should be applied during cross validation, only on the training set
for key in datasets.keys():
    datasets[key]["X"][:, :, 0] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 0], np.min(datasets[key]["X"][:, :, 0])), 
        np.max(datasets[key]["X"][:, :, 0]) - np.min(datasets[key]["X"][:, :, 0])
        )
    
    datasets[key]["X"][:, :, 1] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 1], np.min(datasets[key]["X"][:, :, 1])), 
        np.max(datasets[key]["X"][:, :, 1]) - np.min(datasets[key]["X"][:, :, 1])
        )
    
    datasets[key]["X"][:, :, 2] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 2], np.min(datasets[key]["X"][:, :, 2])), 
        np.max(datasets[key]["X"][:, :, 2]) - np.min(datasets[key]["X"][:, :, 2])
        )
    
    datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
    datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])



In [ ]:
# Applying mappings to y columns
try:
    for key in datasets.keys():
        datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
        datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])
except KeyError as e:
    print(f"Mapping Error; Already Mapped: {e}")

KeyError: <numpy.vectorize object at 0x0000016161C6FCE0>

Initialize DL framework